# 03 · 微调后:加载、对比、合并

> 前置:需要先跑完 [`01`](01_PEFT与LoRA原理与实战.ipynb),它会在 `../outputs/lora-qwen1.5b` 生成 LoRA 适配器。

本 notebook 做三件事:

1. **加载适配器**:底座 + 适配器 = 微调后的模型。
2. **前后对比**:同一个问题,微调前 vs 微调后的回答差异(重点看有没有学会「—— 你的学习小助手」这个风格)。
3. **合并权重**:`merge_and_unload` 把 LoRA 融进底座,得到一个可独立部署的完整模型。

## 一、加载底座并定义生成函数

先加载 bf16 的底座(不带适配器),这就是「微调前」的模型。

In [ ]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "../outputs/lora-qwen1.5b"
SYSTEM_PROMPT = "你是一个热心的中文学习小助手。"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda",
)
base_model.eval()


@torch.no_grad()
def chat(model, question, max_new_tokens=128):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    out = model.generate(
        inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()


print("底座已加载,生成函数就绪。")

## 二、加载 LoRA 适配器 =「微调后」模型

`PeftModel.from_pretrained(底座, 适配器目录)` 就能把适配器挂回去。
底座还是同一份权重,只是多了训练好的 LoRA 分支。

In [ ]:
from peft import PeftModel

# 注意:from_pretrained 会在传入的 base_model 上原地挂载适配器
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
tuned_model.eval()
print("LoRA 适配器已加载,得到微调后的模型。")

## 三、前后对比

同一批问题,分别看**微调前**(用 `disable_adapter()` 临时关掉 LoRA,等价于纯底座)和**微调后**的回答。

重点观察:微调后的回答是否更像我们数据里的风格,尤其是结尾的「—— 你的学习小助手」签名。

> 提示:如果只跑了演示的 60 步,风格可能只是「初见端倪」。想效果明显,请回到 `01` 把 `max_steps` 调大后重训。

In [ ]:
questions = [
    "什么是过拟合?",
    "给我三条学习编程的建议。",
    "用一句话解释什么是微调。",
]

for q in questions:
    # 微调前:临时禁用 LoRA 适配器,等价于原始底座
    with tuned_model.disable_adapter():
        before = chat(tuned_model, q)
    # 微调后:启用适配器
    after = chat(tuned_model, q)

    print("=" * 70)
    print(f"【问题】{q}")
    print(f"\n[微调前]\n{before}")
    print(f"\n[微调后]\n{after}")
print("=" * 70)

## 四、合并权重(merge_and_unload)

推理时「底座 + 适配器」会有一点点额外开销。若要长期部署,可以把 LoRA 增量**永久融合**进底座权重,得到一个普通的完整模型,之后加载就不再依赖 `peft`。

$$
W_{\text{merged}} = W + \frac{\alpha}{r} BA
$$

> 重要:合并要在**高精度底座(bf16/fp16)**上做,像这里的 1.5B LoRA。
> `02` 的 QLoRA 底座是 4-bit,**不能**直接合并,需要先用 fp16 重新加载同名底座、再挂适配器合并。

In [ ]:
MERGED_DIR = "../outputs/merged-qwen1.5b"

# merge_and_unload:把 LoRA 融进底座,返回一个普通的 transformers 模型
merged_model = tuned_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"已合并并保存完整模型到: {MERGED_DIR}")

# 验证合并后的模型仍能正常回答
print("\n合并后模型测试:")
print(chat(merged_model, "什么是神经网络?"))

## 小结

- **加载**:`PeftModel.from_pretrained(底座, 适配器目录)`;`disable_adapter()` 可临时回到纯底座。
- **对比**:通过前后回答判断微调是否生效;我们用「—— 你的学习小助手」签名作为易观察的风格标记。
- **合并**:`merge_and_unload()` 得到独立完整模型,方便部署;QLoRA 的 4-bit 底座需先换 fp16 底座再合并。
- **评估**:这里是主观对比,真实项目还会用困惑度、任务指标、或用更强的模型打分等更客观的方式。

至此,LoRA / QLoRA / PEFT 的完整链路(原理 → 微调 → 部署)就跑通了。
最后一站:[`04_衍生_从SFT走向强化学习`](04_衍生_从SFT走向强化学习.ipynb),看看 SFT 的天花板在哪、为什么还需要强化学习。